# 01 — PDF matnini ajratib olish (PyMuPDF)

**Milestone:** M2 · **Task:** M2.T1 · **Dependency:** M1

## Maqsad
PDF fayldan sahifa-darajasidagi matnni olish va uni ishonchli qilish:
qaysi PDF o'qiladi, qaysi biri yo'q, qaysi biri skanerlangan.

## O'rganish savollari
1. PDF "matn qatlami" nima? Skanerlangan PDF'da nega matn yo'q?
2. `page.get_text("text")` va `page.get_text("blocks")` farqi nimada?
3. Nima uchun `.pdf` kengaytmasiga ishonish yetarli emas?

## Bu notebook nima qilmaydi
Production kod **keyin** yoziladi: `app/services/pdf_extractor.py`.
Bu yerda faqat **o'rganish** — xatolarni qo'lda ushlab, keyin
implementatsiyaga aynantiriladi.

## 0. Tayyorlov

In [1]:

import os
import sys
from pathlib import Path

# Notebook `notebooks/` ichida saqlanadi, lekin kernel cwd si qayerdan
# boshlanishiga qarab farq qilishi mumkin (nbconvert default: notebook
# papkasi). Shuning uchun loyiha ildizini **belgi fayl bo'yicha** qidiriamiz
# — hard-coded `..` ishlamaydi.
ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/make_sample_pdfs.py").exists()
)
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pymupdf

print("PyMuPDF:", pymupdf.__doc__.splitlines()[0])
print("Loyiha ildizi:", ROOT)
print("Notebook cwd :", Path.cwd())

SAMPLES = sorted(Path("data/raw").glob("*.pdf"))
if not SAMPLES:
    raise SystemExit(
        "data/raw/ bo'sh. Namuna PDF'larni yaratish uchun: "
        ".venv/bin/python scripts/make_sample_pdfs.py"
    )

for p in SAMPLES:
    print(f"  {p.name:38} {p.stat().st_size:>7} bayt")


PyMuPDF: PyMuPDF 1.28.2: Python bindings for the MuPDF 1.28.2 library.
Loyiha ildizi: /home/lutfullo/education/ml/unsupervised-learn-
Notebook cwd : /home/lutfullo/education/ml/unsupervised-learn-
  01_short_ml_basics.pdf                  791191 bayt
  02_medium_distributed_systems.pdf       795896 bayt
  03_multi_topic_report.pdf               799007 bayt
  04_mixed_language.pdf                   793617 bayt


## 1. PDF ochish — `pymupdf.open()`

**O'rganish:** `Document` obyekti PDF faylni ifodalaydi. Uni ochganimizda
fayl **butunlay xotiraga yuklanmaydi** — sahifalar ker bo'lganda o'qiladi.

Muhim atributlar:

| Atribut | Ma'nosi |
|---------|---------|
| `len(doc)` | sahifalar soni |
| `doc.metadata` | PDF ichidagi `Title`, `Author`, `Producer` … |
| `doc.needs_pass` | parol talab qilinadimi |
| `doc.is_pdf` | bu haqiqatan PDF formatidami |

In [2]:

doc = pymupdf.open(SAMPLES[0])
print(f"Fayl      : {SAMPLES[0].name}")
print(f"Sahifalar : {len(doc)}")
print(f"is_pdf    : {doc.is_pdf}")
print(f"needs_pass: {bool(doc.needs_pass)}")

print("\nmetadata (bo'sh qiymatlar tushirib qoldirilgan):")
for key, value in doc.metadata.items():
    if value:
        print(f"  {key:10} = {value}")


Fayl      : 01_short_ml_basics.pdf
Sahifalar : 3
is_pdf    : True
needs_pass: False

metadata (bo'sh qiymatlar tushirib qoldirilgan):
  format     = PDF 1.7


> **Kuzatuv:** Namuna PDF'larimizda `metadata` de faqat `format` to'ldirilgan.
> Bu normal — `reportlab`/`pymupdf` yaratuvchilar `Title`/`Author` yozmaydi.
> **Ishonchni metadata'ga emas, haqiqiy matnga bog'lash kerak.**

## 2. Sahifalarni aylanib chiqish

**O'rganish:** `doc` obyekti **iterable** — `for page in doc` ishlaydi.
Sahifa raqamini **qo'lda sanash shart emas**, lekin odam ko'radigan raqam
**1-based** bo'lishi kerak (`docs/io_contract.md` §2.1).

In [3]:

pages = []
for i, page in enumerate(doc, start=1):
    text = page.get_text("text").strip()
    pages.append((i, len(text), text))

print(f"{'Sahifa':>7} {'Belgi':>7}  {'Birinchi 60 belgi'}")
print("-" * 78)
for num, n, text in pages:
    print(f"{num:>7} {n:>7}  {text[:60]!r}")

total = sum(n for _, n, _ in pages)
print(f"\nJami: {total} belgi, {len(pages)} sahifa")
print(f"O'rtacha: {total / len(pages):.0f} belgi/sahifa")


 Sahifa   Belgi  Birinchi 60 belgi
------------------------------------------------------------------------------
      1     665  'Chapter 1. Introduction to Supervised Learning\n============='
      2     755  'Chapter 2. Unsupervised Learning and Clustering\n============'
      3     645  'Chapter 3. Evaluation of Unsupervised Models\n==============='

Jami: 2065 belgi, 3 sahifa
O'rtacha: 688 belgi/sahifa


## 3. `get_text("text")` va `get_text("blocks")` farqi

**O'rganish:** `text` — sahifa matnini **oddiy qatorlar ko'rinishida** beradi.
`blocks` — matnni **mantiqiy bloklarga** (sarlavha, paragraf, jadval) bo'ladi
va har bir blokning bbox (joylashuv) ma'lumoti bor.

Bu farq **keyin muhim**: chunk'lash (M4) va TF-IDF (M5) uchun bloklar
ko'proq ma'lumot beradi (sarlavha blokini alohida ko'rib chiqish mumkin).

Uchinchi variant — `sort=True`: bloklarni **yuqoridan pastga, chapdan o'ngga**
tartibda chiqaradi. Bu **ikki ustunli** PDF'lar uchun kerak.

In [4]:

page = doc[0]

variants = {
    'get_text("text")': page.get_text("text"),
    'get_text("text", sort=True)': page.get_text("text", sort=True),
    'get_text("blocks")': repr(page.get_text("blocks"))[:400],
}

for name, out in variants.items():
    print(f"{name:36} uzunligi = {len(out):>5}")

print("\n--- blocks tuzilishi (birinchi 2 ta blok) ---")
for block in page.get_text("blocks")[:2]:
    x0, y0, x1, y1, content, block_no, block_type = block[:7]
    kind = "MATN" if block_type == 0 else f"type={block_type}"
    print(f"  blok {block_no} [{kind}] bbox=({x0:.0f},{y0:.0f})-({x1:.0f},{y1:.0f})")
    print(f"    {content.strip()[:70]!r}")

print("\n--- sort=True taqqoslash (2-ustunli bo'lmagan hujjatda) ---")
a, b = page.get_text("text"), page.get_text("text", sort=True)
print("Bir xilmi:", a == b, "| uzunlik farqi:", len(b) - len(a))


get_text("text")                     uzunligi =   666
get_text("text", sort=True)          uzunligi =   667
get_text("blocks")                   uzunligi =   400

--- blocks tuzilishi (birinchi 2 ta blok) ---
  blok 0 [MATN] bbox=(60,60)-(493,85)
    'Chapter 1. Introduction to Supervised Learning\n======================='
  blok 1 [MATN] bbox=(60,96)-(530,180)
    'Supervised learning trains a model on labelled examples. The dataset c'

--- sort=True taqqoslash (2-ustunli bo'lmagan hujjatda) ---
Bir xilmi: False | uzunlik farqi: 1


## 4. Bo'sh sahifalarni aniqlash

**O'rganish:** `text.strip() == ""` bo'lsa — sahifa **matnsiz**. Bu
kutilgan holat: muqova sahifasi, ajratgich, rasm.

**Muhim qaror:** bo'sh sahifalarni **o'chirmaymiz** — ular statistik uchun
kerak (`docs/io_contract.md` §2.1: "Bo'sh sahifalar o'chirilmaydi").

In [5]:

print(f"{'Fayl':38} {'sahifa':>7} {'belgi':>7} {'bo\'sh sahifa':>13}")
print("-" * 70)

for path in SAMPLES:
    with pymupdf.open(path) as d:
        texts = [p.get_text("text").strip() for p in d]
    empty = [i for i, t in enumerate(texts, start=1) if not t]
    print(
        f"{path.name:38} {len(texts):>7} {sum(map(len, texts)):>7} "
        f"{(str(empty) if empty else 'yo\'q'):>13}"
    )


Fayl                                    sahifa   belgi  bo'sh sahifa
----------------------------------------------------------------------
01_short_ml_basics.pdf                       3    2065          yo'q
02_medium_distributed_systems.pdf            8    4920          yo'q
03_multi_topic_report.pdf                   12    6443          yo'q
04_mixed_language.pdf                        6    3036          yo'q


## 5. Xatolarni ushlab olish

To'rtta holatni **qo'lda** tekshiramiz — keyin har biri alohida testga aylanadi.

| Holat | Nima bo'ladi | Kerakli xato |
|-------|--------------|--------------|
| Fayl yo'q | `FileNotFoundError` | `InvalidPdfError` |
| Buzilgan PDF | `pymupdf.FileDataError` | `InvalidPdfError` |
| Parol bilan himoyalangan | `needs_pass == True` | `InvalidPdfError` |
| `.pdf` nomli PNG | `pymupdf.FileDataError` | `InvalidPdfError` |

In [6]:

import tempfile

tmp = Path(tempfile.mkdtemp(prefix="m2_t1_"))
print("Fixture papkasi:", tmp)


def make_scanned_pdf(dest: Path) -> Path:
    """Matn qatlami yo'q PDF — raster rasm (skanerlangan kabi)."""
    out = pymupdf.open()
    pix = pymupdf.Pixmap(pymupdf.csRGB, pymupdf.IRect(0, 0, 400, 400))
    pix.set_rect(pix.irect, (170, 170, 180))
    for _ in range(2):
        page = out.new_page()
        page.insert_image(pymupdf.Rect(0, 0, 400, 400), pixmap=pix)
    out.save(dest)
    out.close()
    return dest


def make_encrypted_pdf(src: Path, dest: Path, pw: str) -> Path:
    with pymupdf.open(src) as d:
        dest.write_bytes(
            d.tobytes(
                encryption=pymupdf.PDF_ENCRYPT_AES_256,
                owner_pw="owner",
                user_pw=pw,
            )
        )
    return dest


# 1) fayl yo'q
# 2) buzilgan PDF — birinchi 300 bayt
corrupted = tmp / "corrupted.pdf"
corrupted.write_bytes(SAMPLES[0].read_bytes()[:300])
# 3) parol bilan himoyalangan
encrypted = make_encrypted_pdf(SAMPLES[0], tmp / "encrypted.pdf", "secret")
# 4) .pdf nomli PNG — kengaytmaga ishonib qolmaymiz
fake = tmp / "fake.pdf"
fake.write_bytes(pymupdf.Pixmap(pymupdf.csRGB, pymupdf.IRect(0, 0, 60, 60)).tobytes("png"))
# 5) skanerlangan
scanned = make_scanned_pdf(tmp / "scanned.pdf")

for name, path in [
    ("buzilgan", corrupted),
    ("parol bilan", encrypted),
    ("PNG -> .pdf", fake),
    ("skanerlangan", scanned),
]:
    print(f"  {name:14} {path.name:16} {path.stat().st_size:>6} bayt")


Fixture papkasi: /tmp/m2_t1_hmxee4id


  buzilgan       corrupted.pdf       300 bayt
  parol bilan    encrypted.pdf    791993 bayt
  PNG -> .pdf    fake.pdf           2091 bayt
  skanerlangan   scanned.pdf      483877 bayt


In [7]:

def probe(path: Path) -> str:
    """Har bir holatni ochib, PyMuPDF qanday javob berishini ko'rsatadi."""
    try:
        with pymupdf.open(path) as d:
            needs_pass = bool(d.needs_pass)
            chars = 0
            if not needs_pass:
                chars = sum(len(p.get_text("text").strip()) for p in d)
            magic = path.read_bytes()[:5]
            return (
                f"ochildi | magic={magic!r} | needs_pass={needs_pass} "
                f"| sahifa={len(d)} | belgi={chars}"
            )
    except Exception as exc:
        return f"XATO {type(exc).__name__}: {str(exc)[:70]}"


print("=== 0) fayl yo'q ===")
print(probe(tmp / "yoq_ma_yoq.pdf"))

for label, path in [
    ("1) buzilgan PDF", corrupted),
    ("2) parol bilan", encrypted),
    ("3) PNG nomi .pdf", fake),
    ("4) skanerlangan", scanned),
    ("5) normal", SAMPLES[0]),
]:
    print(f"\n=== {label} ===")
    print(probe(path))


=== 0) fayl yo'q ===
XATO FileNotFoundError: no such file: '/tmp/m2_t1_hmxee4id/yoq_ma_yoq.pdf'

=== 1) buzilgan PDF ===
XATO FileDataError: Failed to open file '/tmp/m2_t1_hmxee4id/corrupted.pdf' as type pdf.

=== 2) parol bilan ===


ochildi | magic=b'%PDF-' | needs_pass=True | sahifa=3 | belgi=0

=== 3) PNG nomi .pdf ===
ochildi | magic=b'\x89PNG\r' | needs_pass=False | sahifa=1 | belgi=0

=== 4) skanerlangan ===
ochildi | magic=b'%PDF-' | needs_pass=False | sahifa=2 | belgi=0

=== 5) normal ===
ochildi | magic=b'%PDF-' | needs_pass=False | sahifa=3 | belgi=2065


### Kutilgan natija va xulosa

1. **Fayl yo'q** → `FileNotFoundError` (path turi noto'g'ri bo'lsa)
2. **Buzilgan** → `pymupdf.FileDataError` ("cannot open broken document")
3. **Parol bilan** → `open()` **muvaffaqiyatli**, lekin `needs_pass == True`.
   Matn **bo'sh** qaytariladi — ya'ni "parol bilan" va "skanerlangan"
   holatlari **aralashib ketadi**. Shu sabab `needs_pass` **alohida**
   tekshirilishi shart.
4. **`.pdf` nomli PNG** → `pymupdf.FileDataError`. Kengaytma tekshiruvi
   ishlagan, lekin **nima uchun** faqat kengaytmaga ishongandan aniqroq
   xatoni beradi — shu sabab **magic bytes** `%PDF` tekshiriladi
   (`docs/io_contract.md` §1.2, I1).
5. **Skanerlangan** → ochiladi, `len(doc) > 0`, lekin belgi = **0**.
   Bu holat `ScannedPdfError` beradi (I4).

## 6. Magic bytes — nima uchun kerak

`docs/io_contract.md` §1.2 **I1** — "eng muhim qoida".

```text
PDF fayl birinchi 5 bayti  ->  %PDF-      (spec: %PDF-1.7, %PDF-2.0, ...)
PNG fayl birinchi 8 bayti  ->  \x89PNG\r\n\x1a\n
```

Kengaytma `.pdf` **ishonchli emas**: foydalanuvchi `test.pdf` nomli PNG
yuborsa, kengaytma hech narsa aytmaydi. Magic bytes esa **tarkibni**
tekshiradi va xatoni **oldindan** aniqroq qiladi.

> **Texnik nuqta:** qiyoslash `magic == b"%PDF"` emas, `magic.startswith(b"%PDF")`
> bo'lishi kerak — `%PDF` **4** belgi, tekshiriladigan **5 bayt** esa `%PDF-`.

In [8]:

print(f"{'Fayl':20} {'magic (5 bayt)':22} {'%PDF?':>7}")
print("-" * 52)

for label, path in [
    ("haqiqiy PDF", SAMPLES[0]),
    ("parol bilan", encrypted),
    ("buzilgan PDF", corrupted),
    ("PNG -> .pdf", fake),
    ("skanerlangan", scanned),
]:
    magic = path.read_bytes()[:5]
    print(f"{label:20} {magic!r:22} {str(magic.startswith(b'%PDF')):>7}")

print("\nXulosa:")
print("  * haqiqiy / parol-bilan / skanerlangan — hammasi `%PDF-`:")
print("    magic bytes **formatni** tekshiradi, **holatni** esa qilmaydi.")
print("  * buzilgan PDF ham `%PDF-` (header butun, jism buzilgan) —")
print("    uni faqat `pymupdf.open()` ushlaydi.")
print("  * PNG -> `.pdf` — magic bilan **aniq** ushlanadi.")


Fayl                 magic (5 bayt)           %PDF?
----------------------------------------------------
haqiqiy PDF          b'%PDF-'                  True
parol bilan          b'%PDF-'                  True
buzilgan PDF         b'%PDF-'                  True
PNG -> .pdf          b'\x89PNG\r'             False
skanerlangan         b'%PDF-'                  True

Xulosa:
  * haqiqiy / parol-bilan / skanerlangan — hammasi `%PDF-`:
    magic bytes **formatni** tekshiradi, **holatni** esa qilmaydi.
  * buzilgan PDF ham `%PDF-` (header butun, jism buzilgan) —
    uni faqat `pymupdf.open()` ushlaydi.
  * PNG -> `.pdf` — magic bilan **aniq** ushlanadi.


## 7. Skanerlangan PDF aniqlash

**Qoida (I4):** barcha sahifalardagi jami belgi < `MIN_TOTAL_CHARS` (100)
→ `ScannedPdfError`.

**Nega 100?** juda qisqa hujjat (masalan 1 jadval) ham skanerlangan
bo'lmasligi mumkin. chegara juda past bo'lsa — **oddiy PDF rad etiladi**,
juda yuqori bo'lsa — **skanerlangan PDF o'tib ketadi**. 100 — ikkala
xatodan ham qat'iy.

Muhim: **sahifalarni o'chirmaymiz**, avval jami belgi hisoblaymiz.

In [9]:

MIN_TOTAL_CHARS = 100

print(f"{'Fayl':20} {'belgi':>7} {'chegara':>8}  Xulosa")
print("-" * 58)

for label, path in [
    *[(p.name[:19], p) for p in SAMPLES],
    ("skanerlangan", scanned),
]:
    with pymupdf.open(path) as d:
        chars = sum(len(p.get_text("text").strip()) for p in d)
    verdict = "MATN BOR" if chars >= MIN_TOTAL_CHARS else "SCANNED -> xato"
    print(f"{label:20} {chars:>7} {MIN_TOTAL_CHARS:>8}  {verdict}")


Fayl                   belgi  chegara  Xulosa
----------------------------------------------------------
01_short_ml_basics.     2065      100  MATN BOR
02_medium_distribut     4920      100  MATN BOR
03_multi_topic_repo     6443      100  MATN BOR
04_mixed_language.p     3036      100  MATN BOR
skanerlangan               0      100  SCANNED -> xato


## 8. Ikki ustunli PDF — `sort=True` **yordam bermaydi**

**O'rganish:** `sort=True` matnni **vertikal koordinat** bo'yicha saralaydi.
Ikki ustunli hujjatda bu chap va o'ng ustunni **qatorlab aralashib** ketiradi.

Quyida ikki hujjat yaratilib, ikkala variant ham sinab ko'riladi:

- **A** — ustunlar ketma-ket yozilgan (real hujjatga o'xshash)
- **B** — qatorlar aralash yozilgan

In [10]:

def make_two_column(dest: Path, interleaved: bool) -> Path:
    """Ikki ustunli PDF. `interleaved=True` — qatorlar aralash yozilgan."""
    out = pymupdf.open()
    page = out.new_page()
    if interleaved:
        for i in range(6):
            y = 70 + i * 14
            page.insert_text((50, y), f"L{i}", fontname="helv", fontsize=10)
            page.insert_text((310, y), f"R{i}", fontname="helv", fontsize=10)
    else:
        page.insert_textbox(
            pymupdf.Rect(50, 60, 290, 760), "LEFT\n" * 8, fontname="helv", fontsize=9
        )
        page.insert_textbox(
            pymupdf.Rect(310, 60, 550, 760), "RIGHT\n" * 8, fontname="helv", fontsize=9
        )
    out.save(dest)
    out.close()
    return dest


doc_a = make_two_column(tmp / "two_col.pdf", interleaved=False)
doc_b = make_two_column(tmp / "interleaved.pdf", interleaved=True)

for label, path in [("A — ketma-ket", doc_a), ("B — aralash", doc_b)]:
    page = pymupdf.open(path)[0]
    plain = " ".join(page.get_text("text").split())
    srt = " ".join(page.get_text("text", sort=True).split())
    print(f"=== {label} ===")
    print(f"  sort=False: {plain[:64]}")
    print(f"  sort=True : {srt[:64]}\n")


=== A — ketma-ket ===
  sort=False: LEFT LEFT LEFT LEFT LEFT LEFT LEFT LEFT RIGHT RIGHT RIGHT RIGHT 
  sort=True : LEFT RIGHT LEFT RIGHT LEFT RIGHT LEFT RIGHT LEFT RIGHT LEFT RIGH

=== B — aralash ===
  sort=False: L0 R0 L1 R1 L2 R2 L3 R3 L4 R4 L5 R5
  sort=True : L0 R0 L1 R1 L2 R2 L3 R3 L4 R4 L5 R5



In [11]:

# A hujjatida aniq o'lchov: birinchi RIGHT dan keyin yana LEFT keladimi?
page = pymupdf.open(doc_a)[0]
plain, srt = page.get_text("text"), page.get_text("text", sort=True)

for name, txt in [("sort=False", plain), ("sort=True", srt)]:
    interleaved = txt.rfind("LEFT") > txt.find("RIGHT")
    verdict = "BUZILGAN" if interleaved else "TO'G'RI"
    print(f"{name:11} aralashganmi: {str(interleaved):5}  -> {verdict}")

print("\nXulosa: sort=True A holatni BUZADI va B holatni tuzatMAYDI.")
print("         -> sort=True ishlatilmaydi (docs/limitations.md L-01).")


sort=False  aralashganmi: False  -> TO'G'RI
sort=True   aralashganmi: True   -> BUZILGAN

Xulosa: sort=True A holatni BUZADI va B holatni tuzatMAYDI.
         -> sort=True ishlatilmaydi (docs/limitations.md L-01).


## 9. Ishlab chiqilgan modul bilan tekshirish

Yuqoragi kichik tajriba endi **production kod** bilan qaydalanadi.
`app/services/pdf_extractor.py` hujjat yuqoridagi xulosalarga to'liq
amal qiladi: magic bytes, `needs_pass`, matn chegarasi, bo'sh sahifalarni
saqlash.

In [12]:

from app.core.exceptions import InvalidPdfError, PdfAnalyzerError, ScannedPdfError
from app.services import extract_pages

print(f"{'Namuna PDF':34} {'sahifa':>7} {'belgi':>7}  Holat")
print("-" * 64)
for path in SAMPLES:
    pages = extract_pages(path)
    print(
        f"{path.name:34} {len(pages):>7} {sum(p.n_chars for p in pages):>7}  ✅ o'qildi"
    )

print("\n=== Xatolar moduli darajasida ===")
for label, path in [
    ("yo'q fayl", tmp / "nope.pdf"),
    ("buzilgan", corrupted),
    ("parol bilan", encrypted),
    ("PNG -> .pdf", fake),
    ("skanerlangan", scanned),
]:
    try:
        extract_pages(path)
        print(f"  {label:14} -> ❌ xato berilmadi")
    except PdfAnalyzerError as exc:
        print(f"  {label:14} -> {type(exc).__name__}: {str(exc)[:50]}")


Namuna PDF                          sahifa   belgi  Holat
----------------------------------------------------------------
01_short_ml_basics.pdf                   3    2065  ✅ o'qildi
02_medium_distributed_systems.pdf        8    4920  ✅ o'qildi
03_multi_topic_report.pdf               12    6443  ✅ o'qildi
04_mixed_language.pdf                    6    3036  ✅ o'qildi

=== Xatolar moduli darajasida ===
  yo'q fayl      -> InvalidPdfError: Fayl topilmadi: /tmp/m2_t1_hmxee4id/nope.pdf
  buzilgan       -> InvalidPdfError: PDF faylni ochib bo'lmadi: /tmp/m2_t1_hmxee4id/cor
  parol bilan    -> InvalidPdfError: PDF parol bilan himoyalangan. Parol bilan ochish q
  PNG -> .pdf    -> InvalidPdfError: /tmp/m2_t1_hmxee4id/fake.pdf: PDF formatida emas —
  skanerlangan   -> ScannedPdfError: Matn qatlami deyarli yo'q: 0 belgi < 100. Bu skane


In [13]:

# Bo'sh sahifalar saqlanadimi? (muhim: statistika uchun kerak)
with_blank = make_two_column  # noqa: F841  (quyidagi misol uchun)

doc = pymupdf.open()
for i, content in enumerate(["", "Second page has real content. " * 10, ""]):
    doc.new_page().insert_textbox(
        pymupdf.Rect(72, 72, 523, 770), content, fontname="helv", fontsize=12
    )
blank_mix = tmp / "blank_mix.pdf"
doc.save(blank_mix)
doc.close()

pages = extract_pages(blank_mix)
print("sahifalar:", [p.page for p in pages])
print("bo'shmi:  ", [p.is_empty for p in pages])
print("\nBo'sh sahifalar O'CHIRILMADI — raqamlari saqlanib qoldi ✅")


sahifalar: [1, 2, 3]
bo'shmi:   [True, False, True]

Bo'sh sahifalar O'CHIRILMADI — raqamlari saqlanib qoldi ✅


## 10. Umumiy xulosa — natija jadvali

| Namuna PDF | Sahifa | Belgi | Holat |
|------------|--------|-------|-------|
| `01_short_ml_basics.pdf` | 3 | 2 065 | ✅ o'qiladi |
| `02_medium_distributed_systems.pdf` | 8 | 4 920 | ✅ o'qiladi |
| `03_multi_topic_report.pdf` | 12 | 6 443 | ✅ o'qiladi |
| `04_mixed_language.pdf` | 6 | 3 036 | ✅ o'qiladi (561 Kirill) |

## Topilgan cheklovlar

| ID | Cheklov | Qaror |
|----|---------|-------|
| **L-01** | `sort=True` ikki ustunli matnni buzadi | `sort=True` ishlatilmaydi |
| **L-02** | Skanerlangan PDF | `ScannedPdfError` (OCR non-goal) |
| **L-03** | Parol bilan PDF | `InvalidPdfError` (`needs_pass`) |
| **L-04** | `MIN_TOTAL_CHARS` chegarasi | 100, `.env` dan o'qiladi |

Batafsil: `docs/limitations.md`.

## Keyingi qadam — M3

1. Xom matnni tozalash (`app/nlp/preprocessing.py`)
2. ⚠️ **Texnik atamalarni o'chirmaslik:** `AI`, `API`, `SQL`, `CNN`, `RAG`
3. `display_text` (odam uchun) va `ml_text` (vektor uchun) ajratish

> Eslatma: bu notebook `data/raw/` ga bog'liq — u `.gitignore`da,
> shuning uchun **testlar** bu fayllarga bog'liq emas
> (`docs/limitations.md` L-05).